Step 1: Import Pandas

In [1]:
import pandas as pd
from pathlib import Path

print("Pandas version:", pd.__version__)

Pandas version: 3.0.5


Step 2: Import the whole folder of data

In [2]:
from pathlib import Path

data_path = Path("data")

files = list(data_path.rglob("*.*"))

for file in files:
    print(file)

data\customers\dim_customer.csv
data\inventory\fact_inventory_snapshot.csv
data\products\dim_product.csv
data\promotions\dim_promotion.csv
data\sales\sales_2025-01.csv
data\sales\sales_2025-02.csv
data\sales\sales_2025-03.csv
data\sales\sales_2025-04.csv
data\sales\sales_2025-05.csv
data\sales\sales_2025-06.csv
data\sales\sales_2025-07.csv
data\sales\sales_2025-08.csv
data\sales\sales_2025-09.csv
data\sales\sales_2025-10.csv
data\sales\sales_2025-11.csv
data\sales\sales_2025-12.csv
data\sales\sales_2026-01.csv
data\sales\sales_2026-02.csv
data\sales\sales_2026-03.csv
data\sales\sales_2026-04.csv
data\sales\sales_2026-05.csv
data\sales\sales_2026-06.csv
data\sales\sales_2026-07.csv
data\sales\sales_2026-08.csv
data\sales\sales_2026-09.csv
data\sales\sales_returns.csv
data\salesforce\dim_salesforce.csv
data\targets\fact_target.csv


Step 3: Audit files

In [3]:
audit_results = []

for file in files:

    if file.suffix.lower() == ".csv":
        df_temp = pd.read_csv(file)

    elif file.suffix.lower() in [".xlsx", ".xls"]:
        df_temp = pd.read_excel(file)

    else:
        continue

    audit_results.append({
        "File": file.name,
        "Folder": file.parent.name,
        "Rows": len(df_temp),
        "Columns": len(df_temp.columns),
        "DuplicateRows": df_temp.duplicated().sum(),
        "MissingValues": df_temp.isna().sum().sum()
    })

audit = pd.DataFrame(audit_results)

audit

,File,Folder,Rows,Columns,DuplicateRows,MissingValues
0,dim_customer.csv,customers,6500,13,0,0
1,fact_inventory_snapshot.csv,inventory,75600,7,0,0
2,dim_product.csv,products,180,9,0,0
3,dim_promotion.csv,promotions,60,8,0,0
4,sales_2025-01.csv,sales,30005,14,0,0
5,sales_2025-02.csv,sales,27355,14,0,0
6,sales_2025-03.csv,sales,30133,14,0,0
7,sales_2025-04.csv,sales,29283,14,0,0
8,sales_2025-05.csv,sales,30256,14,0,0
9,sales_2025-06.csv,sales,29122,14,0,0


Step 4: Evaluate the schema

In [4]:
sales_files = sorted(
    Path("data/sales").glob("sales_20*.csv")
)

print("Number of monthly sales files:", len(sales_files))

for file in sales_files:
    temp = pd.read_csv(file, nrows=5)

    print(file.name)
    print(temp.columns.tolist())
    print()

Number of monthly sales files: 21
sales_2025-01.csv
['TransactionID', 'Date', 'CustomerID', 'ProductID', 'Qty', 'UnitListPrice', 'GrossSales', 'DiscountPct', 'DiscountAmount', 'NetSales', 'COGS', 'GrossMargin', 'OrderType', 'PaymentTermDays']

sales_2025-02.csv
['TransactionID', 'Date', 'CustomerID', 'ProductID', 'Qty', 'UnitListPrice', 'GrossSales', 'DiscountPct', 'DiscountAmount', 'NetSales', 'COGS', 'GrossMargin', 'OrderType', 'PaymentTermDays']

sales_2025-03.csv
['TransactionID', 'Date', 'CustomerID', 'ProductID', 'Qty', 'UnitListPrice', 'GrossSales', 'DiscountPct', 'DiscountAmount', 'NetSales', 'COGS', 'GrossMargin', 'OrderType', 'PaymentTermDays']

sales_2025-04.csv
['TransactionID', 'Date', 'CustomerID', 'ProductID', 'Qty', 'UnitListPrice', 'GrossSales', 'DiscountPct', 'DiscountAmount', 'NetSales', 'COGS', 'GrossMargin', 'OrderType', 'PaymentTermDays']

sales_2025-05.csv
['TransactionID', 'Date', 'CustomerID', 'ProductID', 'Qty', 'UnitListPrice', 'GrossSales', 'DiscountPct', 'D

Step 4.1: make sure the schema is right

In [5]:
reference_columns = pd.read_csv(
    sales_files[0],
    nrows=0
).columns.tolist()

schema_errors = []

for file in sales_files:

    columns = pd.read_csv(
        file,
        nrows=0
    ).columns.tolist()

    if columns != reference_columns:
        schema_errors.append(file.name)

print("Files with schema problems:", schema_errors)

Files with schema problems: []


Step 5: Combine 21 Sales Files

In [6]:
sales_list = []

for file in sales_files:

    temp = pd.read_csv(file)

    temp["SourceFile"] = file.name

    sales_list.append(temp)

fact_sales = pd.concat(
    sales_list,
    ignore_index=True
)

fact_sales.shape

(620000, 15)

Step 6: Data Quality Check

In [7]:
print("Rows:", len(fact_sales))

print(
    "Unique TransactionID:",
    fact_sales["TransactionID"].nunique()
)

print(
    "Duplicate TransactionID:",
    fact_sales["TransactionID"].duplicated().sum()
)

print(
    "Missing TransactionID:",
    fact_sales["TransactionID"].isna().sum()
)

print(
    "Date range:",
    fact_sales["Date"].min(),
    "→",
    fact_sales["Date"].max()
)

Rows: 620000
Unique TransactionID: 620000
Duplicate TransactionID: 0
Missing TransactionID: 0
Date range: 2025-01-01 → 2026-09-30


Step 6.1: Net Sales formula validation

In [8]:
net_sales_error = (
    fact_sales["GrossSales"]
    - fact_sales["DiscountAmount"]
    - fact_sales["NetSales"]
).abs()

print(
    "Net Sales formula errors:",
    (net_sales_error > 1).sum()
)

Net Sales formula errors: 0


Step 6.2: Gross Margin formula validation

In [9]:
margin_error = (
    fact_sales["NetSales"]
    - fact_sales["COGS"]
    - fact_sales["GrossMargin"]
).abs()

print(
    "Gross Margin formula errors:",
    (margin_error > 1).sum()
)

Gross Margin formula errors: 0


Step 7 — Load Dimension and Fact Tables.
Load all remaining datasets to understand their grain, keys and relationships before building the Power BI semantic model.

In [10]:
customers = pd.read_csv("data/customers/dim_customer.csv")
products = pd.read_csv("data/products/dim_product.csv")
targets = pd.read_csv("data/targets/fact_target.csv")
inventory = pd.read_csv("data/inventory/fact_inventory_snapshot.csv")
salesforce = pd.read_csv("data/salesforce/dim_salesforce.csv")
promotions = pd.read_csv("data/promotions/dim_promotion.csv")
returns = pd.read_csv("data/sales/sales_returns.csv")

In [11]:
tables = {
    "Fact Sales": fact_sales,
    "Customers": customers,
    "Products": products,
    "Targets": targets,
    "Inventory": inventory,
    "Salesforce": salesforce,
    "Promotions": promotions,
    "Returns": returns
}

for name, table in tables.items():
    print(f"{name:15} Rows: {len(table):,} | Columns: {len(table.columns)}")

Fact Sales      Rows: 620,000 | Columns: 15
Customers       Rows: 6,500 | Columns: 13
Products        Rows: 180 | Columns: 9
Targets         Rows: 1,728 | Columns: 5
Inventory       Rows: 75,600 | Columns: 7
Salesforce      Rows: 120 | Columns: 6
Promotions      Rows: 60 | Columns: 8
Returns         Rows: 24,000 | Columns: 8


Step 8 — Define Grain

In [12]:
for name, table in tables.items():
    print("\n", "=" * 50)
    print(name)
    print("=" * 50)

    display(table.head(3))


Fact Sales


,TransactionID,Date,CustomerID,ProductID,Qty,UnitListPrice,GrossSales,DiscountPct,DiscountAmount,NetSales,COGS,GrossMargin,OrderType,PaymentTermDays,SourceFile
0,T000192931,2025-01-01,C006056,P0020,8,87000,696000,0.0865,60232.0,635768.0,367816.0,267952.0,Promotion,0,sales_2025-01.csv
1,T000543905,2025-01-01,C001502,P0035,10,112000,1120000,0.0517,57889.0,1062111.0,735610.0,326501.0,Normal,0,sales_2025-01.csv
2,T000112914,2025-01-01,C003632,P0005,8,157000,1256000,0.0000,0.0,1256000.0,858184.0,397816.0,Normal,30,sales_2025-01.csv



Customers


,CustomerID,CustomerName,Region,Province,Channel,SubChannel,CustomerSegment,DistributorID,SalesRepID,CreditLimit,JoinDate,Latitude,Longitude
0,C000001,Outlet 00001,North,Hai Duong,HORECA,Hotel,Bronze,D05,SR013,60000000.0,2026-03-21,19.04691,106.69120
1,C000002,Outlet 00002,South,Ba Ria - Vung Tau,HORECA,Restaurant,Gold,D05,SR041,150000000.0,2023-05-25,17.92707,108.28566
2,C000003,Outlet 00003,North,Ha Noi,GT,Wholesale,Bronze,D13,SR034,100000000.0,2024-02-09,13.20212,108.69557



Products


...

,SalesRepID,SalesRepName,Team,Manager,TenureMonths,MonthlyVisitTarget
0,SR001,Sales Rep 001,Team B,Manager 3,48,184
1,SR002,Sales Rep 002,Team E,Manager 11,72,142
2,SR003,Sales Rep 003,Team F,Manager 12,113,184



Promotions


,PromotionID,PromotionType,Category,Channel,StartDate,EndDate,PromoDiscountPct,Budget
0,PR001,Digital Voucher,Snacks,E-Commerce,2025-04-16,2025-05-07,0.130,27901037
1,PR002,Trade Deal,Ice Cream,MT,2026-01-11,2026-02-18,0.105,74782128
2,PR003,Trade Deal,Ice Cream,GT,2025-07-29,2025-08-08,0.237,34999180



Returns


,ReturnID,ReturnDate,TransactionID,CustomerID,ProductID,ReturnQty,ReturnAmount,ReturnReason
0,R0000001,2025-02-03,T000415847,C000747,P0157,4,676572.0,Customer Rejection
1,R0000002,2026-08-10,T000091106,C001296,P0165,2,24075.0,Expired
2,R0000003,2026-09-24,T000358567,C001114,P0161,4,71710.0,Damaged


Step 9 — Check Primary Key

In [13]:
print("CustomerID unique:",
      customers["CustomerID"].is_unique)

print("ProductID unique:",
      products["ProductID"].is_unique)

print("SalesRepID unique:",
      salesforce["SalesRepID"].is_unique)

print("PromotionID unique:",
      promotions["PromotionID"].is_unique)

print("TransactionID unique:",
      fact_sales["TransactionID"].is_unique)

print("ReturnID unique:",
      returns["ReturnID"].is_unique)

CustomerID unique: True
ProductID unique: True
SalesRepID unique: True
PromotionID unique: True
TransactionID unique: True
ReturnID unique: True


Step 10 — Check Foreign Key

In [14]:
invalid_customers = fact_sales[
    ~fact_sales["CustomerID"].isin(customers["CustomerID"])
]

print("Invalid CustomerID:", len(invalid_customers))

Invalid CustomerID: 0


In [15]:
invalid_products = fact_sales[
    ~fact_sales["ProductID"].isin(products["ProductID"])
]

print("Invalid ProductID:", len(invalid_products))

Invalid ProductID: 0


In [16]:
invalid_salesrep = customers[
    ~customers["SalesRepID"].isin(salesforce["SalesRepID"])
]

print("Invalid SalesRepID:", len(invalid_salesrep))

Invalid SalesRepID: 0


Check unique value

In [17]:
target_key = [
    "Month",
    "Region",
    "Channel",
    "Category"
]

targets.duplicated(
    subset=target_key
).sum()

np.int64(0)

Step 12 — Inventory khác Sales như thế nào?

In [18]:
Step 12 — Inventory khác Sales như thế nào?

Object `nào` not found.


In [19]:
inventory[
    ["SnapshotDate", "DistributorID", "ProductID"]
].duplicated().sum()

np.int64(0)

In [ ]:
Step 13 — Returns cũng có trap

In [20]:
returns.head()

,ReturnID,ReturnDate,TransactionID,CustomerID,ProductID,ReturnQty,ReturnAmount,ReturnReason
0,R0000001,2025-02-03,T000415847,C000747,P0157,4,676572.0,Customer Rejection
1,R0000002,2026-08-10,T000091106,C001296,P0165,2,24075.0,Expired
2,R0000003,2026-09-24,T000358567,C001114,P0161,4,71710.0,Damaged
3,R0000004,2026-03-26,T000536442,C004912,P0084,3,40924.0,Quality Issue
4,R0000005,2025-11-09,T000024977,C004594,P0031,4,196751.0,Quality Issue


In [21]:
invalid_returns = returns[
    ~returns["TransactionID"].isin(
        fact_sales["TransactionID"]
    )
]

print(
    "Returns without original transaction:",
    len(invalid_returns)
)

Returns without original transaction: 0


Step 14 — Promotions là bài khó hơn nữa
Đây gọi là kiểu range/conditional matching.

Chưa xử lý nó ngay. Ghi lại:

### Modeling Risk — Promotion

Promotion cannot be joined to Fact Sales using a simple ProductID foreign key.

Promotion applicability depends on:
- transaction date
- category
- channel

A mapping/bridge or preprocessing logic may be required.

In [22]:
promotions.head()

,PromotionID,PromotionType,Category,Channel,StartDate,EndDate,PromoDiscountPct,Budget
0,PR001,Digital Voucher,Snacks,E-Commerce,2025-04-16,2025-05-07,0.130,27901037
1,PR002,Trade Deal,Ice Cream,MT,2026-01-11,2026-02-18,0.105,74782128
2,PR003,Trade Deal,Ice Cream,GT,2025-07-29,2025-08-08,0.237,34999180
3,PR004,Display,Nutrition,E-Commerce,2025-01-10,2025-02-10,0.172,70596448
4,PR005,Display,Home Care,HORECA,2025-06-02,2025-06-13,0.215,158082876


Step 15 — Bây giờ vẽ model trước khi Power BI Agent làm

                   Dim Product
                       │
                       │
                       ▼
Dim Customer ──────► Fact Sales ◄──── Dim Date
     │
     │
     ▼
Dim SalesForce


Dim Date ───────────► Fact Target
                         ▲
                         │
                Region / Channel /
                    Category


Dim Product ────────► Fact Inventory
                          ▲
                          │
                     Distributor


Fact Sales ─────────► Fact Returns

Step 16 — Tạo Data Model Audit Summary
# Data Model Audit Summary

## Fact Sales
- Grain: One sales transaction
- Primary key: TransactionID
- Customer FK validation: PASS / FAIL
- Product FK validation: PASS / FAIL

## Dim Customer
- Grain: One customer/outlet
- Primary key: CustomerID
- Contains Region, Province, Channel, Distributor and SalesRep

## Dim Product
- Grain: One SKU/product
- Primary key: ProductID

## Dim SalesForce
- Grain: One Sales Representative
- Primary key: SalesRepID

## Fact Target
- Grain: Month × Region × Channel × Category
- Duplicate grain: ...
- Modeling risk: Target must not be duplicated at transaction level

## Fact Inventory
- Grain: Snapshot Date × Distributor × Product
- Modeling risk: Inventory is semi-additive across time

## Fact Returns
- Grain: One return record
- Links to original sales TransactionID

## Promotion
- Grain: One promotion
- Modeling risk: Requires date-range + Category + Channel matching

Từ bước 23 về 25 là tạo src/ và 4 files under srcs. Tạo data frame, business rules, validate data và load data.


Bước 26 — Bây giờ bắt đầu học SQL thật

In [26]:
import duckdb

con = duckdb.connect(
    "database/fmcg_analytics.duckdb"
)

In [31]:
con.sql("""
    SELECT
        COUNT(*) AS Transactions,
        SUM(NetSales) AS NetSales,
        SUM(GrossMargin) AS GrossMargin
    FROM fact_sales
""").show()

┌──────────────┬────────────────┬────────────────┐
│ Transactions │    NetSales    │  GrossMargin   │
│    int64     │     double     │     double     │
├──────────────┼────────────────┼────────────────┤
│       620000 │ 327109162314.0 │ 120055809705.0 │
└──────────────┴────────────────┴────────────────┘



Bước 27 — Đừng JOIN thành một bảng khổng lồ

Đây là phần mình muốn bạn học.

Database nên giữ:

                 dim_product
                     │
                     │
dim_customer ─── fact_sales ─── dim_date
     │
     │
dim_salesforce


fact_target
fact_inventory
fact_returns

Tức là star schema / dimensional model, chứ không phải:

❌ one_big_table.csv

SQL database trở thành analytical layer.

Bước 28 — Tạo dim_date bằng SQL

In [32]:
con.execute("""
CREATE OR REPLACE TABLE dim_date AS

SELECT
    Date,
    YEAR(Date) AS Year,
    MONTH(Date) AS MonthNumber,
    MONTHNAME(Date) AS MonthName,
    QUARTER(Date) AS Quarter,
    YEAR(Date) || '-' ||
        LPAD(MONTH(Date)::VARCHAR, 2, '0') AS YearMonth

FROM (
    SELECT *
    FROM generate_series(
        DATE '2025-01-01',
        DATE '2026-12-31',
        INTERVAL 1 DAY
    )
) t(Date);
""")

print("dim_date created successfully")

dim_date created successfully


In [33]:
con.sql("""
SELECT *
FROM dim_date
LIMIT 10
""").show()

┌─────────────────────┬───────┬─────────────┬───────────┬─────────┬───────────┐
│        Date         │ Year  │ MonthNumber │ MonthName │ Quarter │ YearMonth │
│      timestamp      │ int64 │    int64    │  varchar  │  int64  │  varchar  │
├─────────────────────┼───────┼─────────────┼───────────┼─────────┼───────────┤
│ 2025-01-01 00:00:00 │  2025 │           1 │ January   │       1 │ 2025-01   │
│ 2025-01-02 00:00:00 │  2025 │           1 │ January   │       1 │ 2025-01   │
│ 2025-01-03 00:00:00 │  2025 │           1 │ January   │       1 │ 2025-01   │
│ 2025-01-04 00:00:00 │  2025 │           1 │ January   │       1 │ 2025-01   │
│ 2025-01-05 00:00:00 │  2025 │           1 │ January   │       1 │ 2025-01   │
│ 2025-01-06 00:00:00 │  2025 │           1 │ January   │       1 │ 2025-01   │
│ 2025-01-07 00:00:00 │  2025 │           1 │ January   │       1 │ 2025-01   │
│ 2025-01-08 00:00:00 │  2025 │           1 │ January   │       1 │ 2025-01   │
│ 2025-01-09 00:00:00 │  2025 │         

In [34]:
SELECT
    MIN(Date) AS MinDate,
    MAX(Date) AS MaxDate,
    COUNT(*) AS NumberOfDays
FROM dim_date;

IndentationError: unexpected indent (3996475264.py, line 2)